This notebook aims to import, pre-process and overlay land use and land cover data to produce 6 agroforetsry maps that will be tested systematically using a global georferenced dataset collected through photo-interpretation. The objective of this code is to permit aother researcher to reproduce the results. The data used in this process is as follow : 
- ESRI land use and land cover product (https://doi.org/10.1109/IGARSS47720.2021.9553499)
- ESA WorldCover (https://doi.org/10.5281/ZENODO.7254221)
- Dynamic world (DW) (https://doi.org/10.1038/s41597-022-01307-4)
- Tree Canopy Height from Tolan et al (2024) (https://doi.org/10.1016/j.rse.2023.113888)
- Human Footprint Index (HFI) from Gassert et al., (2023) (https://doi.org/10.3389/frsen.2023.1130896)
- Global Pasture Watch (GPW) from  Parente et al., (2024) (https://doi.org/10.1038/s41597-024-04139-6)
- Global georeferenced validation dataset from Tahiri et al (2026) (https://doi.org/10.21203/rs.3.rs-7961411/v1)

This code was developed with assistance from Claude AI (Anthropic) and has been tested and validated before implementation.

Part 1: Import, preprocess and overlay Land Use and Land cover data

In [ ]:
#Import packages :
import ee
import geemap
# Initilise GEE :
ee.Authenticate()
ee.Initialize(project= 'Your GEE project name')
#Set the vizualization parameters :
Map = geemap.Map(height='700pt', width='700')

Import and pre-process Tree Canopy Height data from Tolan et al (2024)

In [ ]:
#Import using GEE assets link : 
canopyHeight = ee.ImageCollection("projects/meta-forest-monitoring-okw37/assets/CanopyHeight").mosaic()
#Convert tree height into tree / no tree binary mask using a threshold of 1m  whihc corresponds to the minimum tree height used in the validation dataset from Tolan et al (2024)
Trees_higher_1 = canopyHeight.updateMask(canopyHeight.gte(1))
binary_canopy = Trees_higher_1.mask(Trees_higher_1).gt(0).rename('TreeCoverBinary')
#Visualize the binary tree cover map on the map :
Map.addLayer(binary_canopy, {'min': 0, 'max': 1, 'palette': ['red', 'green']}, 'Tolan tree canopy 1m', True)

Import Human Footprint Index data (HFI)  to dilneate grasslands under human pressure ≃ Pasture lands

In [ ]:
# Global 100m Terrestrial Human Footprint (HFP-100) : https://doi.org/10.5061/dryad.ttdz08m1f
# A copy of the data was uploaded to a private GEE asset for processing purposes but is not publicly shared
# The data is publicly available from Dryad: https://doi.org/10.5061/dryad.ttdz08m1f
hpf_100m = ee.ImageCollection("Data link").mosaic()
hpf_100m = hpf_100m.updateMask(hpf_100m.gt(4)) # We set the 4 threshold to only keep areas where human pressure is high accoridng to https://doi.org/10.5061/dryad.ttdz08m1f

Import Land use and land cover maps : 

- 1 ESRI Land Use and Land Cover

In [25]:
# Define a functon to select specifi land use of interest from the ESRI LULC dataset and calculate the mode over a specified year range
def get_lulc_class(dataset_collection, year_range, lulc_class_number, class_name):
    
    # Generate year range
    years = list(range(year_range[0], year_range[1]))
    
    # Function to extract class for a given year
    def extract_class(year):
        dataset = (dataset_collection
                   .filterDate(f'{year}-01-01', f'{year}-12-31')
                   .mosaic())
                   
        class_mask = dataset.eq(lulc_class_number).unmask(0)
        return class_mask.set('year', year).set('class', class_name)
    
    # Create collection of class layers
    class_collection = ee.ImageCollection([extract_class(year) for year in years])
    
    # Calculate mode (most frequent pixel value)
    class_mode = class_collection.reduce(ee.Reducer.mode())
    
    # Mask out non-class areas
    class_masked = class_mode.updateMask(class_mode)
    
    return  class_masked

In [ ]:
# Apply the funtion to croplands ( Id = 5) and rangelands ( Id = 11) for the years 2017 to 2025
esri_lulc = ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')
ESRI_croplands  = get_lulc_class(esri_lulc, (2017, 2024), 5, 'Cropland')

ESRI_rangelands = get_lulc_class(esri_lulc, (2017, 2024), 11, 'Rangeland')
# Apply the HFI threshold to the rangelands to create a pasture mask
ESRI_pastures = ESRI_rangelands.updateMask(hpf_100m_)

# Visualize the results on the map
Map.addLayer(ESRI_croplands, {'palette': ['#ffff00'], 'min': 0, 'max': 9}, 'ESRI croplands')
Map.addLayer(ESRI_pastures, {'palette': ['#ff8000'], 'min': 0, 'max': 9}, 'ESRI Pasture')

 - 2 ESA World Cover

In [ ]:
# ESA cropland mask : 
ESA2021 = ee.ImageCollection('ESA/WorldCover/v200').first()#.clip(ROI)
# Select cropland class (Id = 40) and create a binary mask
ESA_croplands = ESA2021.eq(40)
ESA_croplands = ESA_croplands.updateMask(ESA_croplands)

# Select grassland (Id = 30) and shrubland (Id = 20) classes and create a binary mask
ESA_rangelands = ESA2021.eq(30).Or(ESA2021.eq(20))
ESA_rangelands = ESA_rangelands.updateMask(ESA_rangelands)
# Apply the HFI threshold to the rangelands to create a pasture mask
ESA_pastures = ESA_rangelands.updateMask(hpf_100m_)
# Visualize the ESA cropland and rangeland masks on the map
Map.addLayer(ESA_croplands, {'palette': ['#ffff00'], 'min': 0, 'max': 100}, 'ESA croplands 2020')
Map.addLayer(ESA_pastures, {'palette': ['#00ff00'], 'min': 0, 'max': 100}, 'ESA pastures 2020')

- 3 Dynamic World 


In [ ]:
# Import DW data and filter for the year 2020
dw = (
    ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
    .filterDate('2017-01-01', '2024-12-31')
)

#  Extract the dominant land cover class for the period 2017 - 2024
classification = dw.select('label').mode()

#  Select Croplands = Class 4 (Cropland)
cropland_mask = classification.eq(4)
DW_croplands = cropland_mask.updateMask(cropland_mask)

# #  Select Rangelands = Class 2 (Grass) OR Class 5 (Shrub & Scrub)
rangeland_mask = classification.eq(2).Or(classification.eq(5))
DW_rangelands = rangeland_mask.updateMask(rangeland_mask)
# Apply the HFI threshold to the rangelands to create a pasture mask
DW_pastures = DW_rangelands.updateMask(hpf_100m_)
# # Visualize the ESA cropland and rangeland masks on the map

Map.addLayer(DW_pastures, {'min': 1,'max': 1,'palette': ['#88B053'] } , 'DW Pastures')
Map.addLayer(DW_croplands, {'min': 1,'max': 1,'palette': ['#E49635'] } , 'DW Cropland')

- Global Pasture Watch data

In [ ]:
# Import the full GPW Cultivated Grasslands collection (covers 2000 to present)
gpw_cultivated = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/cultiv-grassland_p")

# Filter the collection to 2017-2024
gpw_cultivated = gpw_cultivated.filterDate('2017-01-01', '2024-12-31')

# Define a function to convert probabilities into 1 and 0
def make_binary(image):
    prob = image.select('probability')
    # Returns 1 where probability >= 50, and 0 where it is < 50
    return prob.gte(50).rename('pasture_binary')

# Map this function over the entire time series collection
binary_collection = gpw_cultivated.map(make_binary)

# Apply the mode reducer 
# For a stack of 1 and 0, the mode will be 1 if it is pasture more than 50% of the time
pasture_mode = binary_collection.reduce(ee.Reducer.mode())

#  Mask out the 0 so we are only left with the stable pasture areas
GPW = pasture_mode.updateMask(pasture_mode.eq(1))

# Visualize the GPW cultivated pasture layer on the map
Map.addLayer(GPW, { 'min': 1,'max': 1,'palette': ['006d2c'] }, ' Cultivated Pasture (>50% of years)')

In [ ]:
# Create agroforestry maps combination :Mosaic of Trees On Croplands (TOC) and Trees On Pastures (TOP) for each of the three LULC datasets (ESRI, ESA, DW) and GPW
# ESRI based agroforestry map :
ESRI_TOC = binary_canopy .updateMask(ESRI_croplands)
ESRI_TOP = binary_canopy .updateMask(ESRI_pastures)
ESRI_ESRI = ee.ImageCollection([ESRI_TOC, ESRI_TOP]).mosaic()
# Visualize the ESRI agroforestry map on the map
Map.addLayer(ESRI_ESRI, {'min': 1,'max': 1,'palette': ['#006d2c'] }, 'ESRI Agroforestry Map')
# ESA based agroforestry map :
ESA_TOC = binary_canopy.updateMask(ESA_croplands)
ESA_TOP = binary_canopy.updateMask(ESA_pastures)
ESA_ESA = ee.ImageCollection([ESA_TOC, ESA_TOP]).mosaic()
# Visualize the ESA agroforestry map on the map
Map.addLayer(ESA_ESA, {'min': 1,'max': 1,'palette': ['#006d2c'] }, 'ESA Agroforestry Map')

# DW based agroforestry map :
DW_TOC = binary_canopy.updateMask(DW_croplands)
DW_TOP = binary_canopy.updateMask(DW_rangelands)
DW_DW = ee.ImageCollection([DW_TOC, DW_TOP]).mosaic()

# GPW based agroforestry map :
GPW_TOP = binary_canopy.updateMask(GPW)
# ESRI x GPW agroforestry map :
ESRI_GPW = ee.ImageCollection([ESRI_TOC, GPW_TOP]).mosaic()
# ESA x GPW agroforestry map :
ESA_GPW = ee.ImageCollection([ESA_TOC, GPW_TOP]).mosaic()
# DW x GPW agroforestry map :
DW_GPW = ee.ImageCollection([DW_TOC, GPW_TOP]).mosaic()

Part 2: Systematic Validation of the combinations using georeferenced dataset collected through Photo-Interpretation From https://doi.org/10.21203/rs.3.rs-7961411/v1

In [ ]:
# Import validation dataset : 
validationPoints = ee.FeatureCollection('projects/ee-ouadyatahiri/assets/ValidationPointsV4') # This is the georferenced dataset used to validate the agroforestry maps

"""
Define the square buffer function: This function examines each validation point and determines whether it overlaps with the agroforestry tree map within a 30m x 30m square buffer.
This buffer size corresponds to the same observation unit dimensions used to collect the validation data.
The function will retuns value 1 if the square buffer around the validation point overlaps with the agroforestry tree map and 0 if it does not. 
This allows us to compare the validation points with the agroforestry maps and calculate accuracy metrics.
The CSV file is exported directly to local drive for further analysis and accuracy metrics calculation in Python.
"""
def square_buffer_points(radius_meters):
    def apply_square_buffer(pt):
        pt = ee.Feature(pt)
        square = pt.buffer(radius_meters).bounds()
        return ee.Feature(square).copyProperties(pt)
    return apply_square_buffer

# # We subdivided the validation points into different groups and iterated over each group to avoid calculation timeout errors in GEE
groups = {
    'group1': ee.Filter.And(ee.Filter.gte('ID', 1), ee.Filter.lte('ID', 2000)),
    'group2': ee.Filter.And(ee.Filter.gte('ID', 2001), ee.Filter.lte('ID', 4000)),
    'group3': ee.Filter.And(ee.Filter.gte('ID', 4001), ee.Filter.lte('ID', 6000)),
    'group4': ee.Filter.And(ee.Filter.gte('ID', 6001), ee.Filter.lte('ID', 8000)),
    'group5': ee.Filter.And(ee.Filter.gte('ID', 8001), ee.Filter.lte('ID', 10000)),
    'group6': ee.Filter.And(ee.Filter.gte('ID', 10001), ee.Filter.lte('ID', 12000)),
    'group7': ee.Filter.And(ee.Filter.gte('ID', 12001), ee.Filter.lte('ID', 14162)),
}

# Parameters to adjust based on the agroforetsry map combination 
MAP_TO_USE = DW_DW  # The agroforestry map combination to test
RADIUS_METERS = 15  # The radius for the square buffer around each validation point (15m corresponds to a 30m x 30m square)
OUTPUT_DIR = " Path" # Output directory for individual group CSVs
FINAL_OUTPUT = "Path + filename" # Final output file path for the combined results of all groups combined

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# List to store all dataframes
all_dfs = []
csv_files = []

# Process each group
for group_name, group_filter in groups.items():
    print(f"Processing {group_name}...")
    
    try:
        # Filter the validation points
        current_group = validationPoints.filter(group_filter)
        
        # Apply square buffer
        square_buffers = current_group.map(square_buffer_points(RADIUS_METERS))
        
        # Extract data from map
        sampled_points = MAP_TO_USE.reduceRegions(
            collection=square_buffers, 
            reducer=ee.Reducer.mode(), 
            scale=1, 
            tileScale=10
        ).getInfo()
        
        # Convert to GeoDataFrame
        zone_stats = gpd.GeoDataFrame.from_features(sampled_points, crs='epsg:4326')
        
        # Count non-null values
        non_null_count = zone_stats['mode'].notnull().sum()
        print(f"{group_name}: {non_null_count} valid points")
        
        # Save individual CSV
        csv_path = os.path.join(OUTPUT_DIR, f"{group_name}_results.csv")
        zone_stats.to_csv(csv_path, index=False)
        
        # Add to list for merging
        all_dfs.append(zone_stats)
        csv_files.append(csv_path)
        
    except Exception as e:
        print(f"Error processing {group_name}: {str(e)}")

# Merge all dataframes
if all_dfs:
    print("\nMerging all groups...")
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    print(f"Total rows in combined dataset: {len(combined_df)}")
    print(f"Total non-null mode values: {combined_df['mode'].notnull().sum()}")
    
    # Save combined CSV
    combined_df.to_csv(FINAL_OUTPUT, index=False)
    print(f"\nFinal combined file saved to: {FINAL_OUTPUT}")
    
else:
    print("No groups were processed successfully")